In [ ]:
"""
================================================================================
PART 1: D_CRYPTOGUARD-MLP GRIDSEARCH ENGINE
GridSearch 432 Combinations × 5-Fold CV with Checkpoint System
================================================================================
"""

import os
import sys
import random
import gc
import time
from itertools import product

import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras import Sequential, regularizers
from keras.layers import Dense, Dropout, BatchNormalization
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import matthews_corrcoef
import matplotlib.pyplot as plt
import seaborn as sns

# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION & PATHS
# ══════════════════════════════════════════════════════════════════════════════

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.config.set_visible_devices([], 'GPU')

DATASET = "D"
DATE = "08082026"
CV_FOLDS = 5
EPOCHS = 100
PATIENCE_ES = 5
PATIENCE_LR = 5

TAG = "D_CryptoGuard-MLP_GridSearchCV_5Fold"
OUT_DIR = f"./0. D_CryptoGuard-MLP_GridSearchCV_5Fold_{DATE}"
os.makedirs(OUT_DIR, exist_ok=True)

FILE_RESULTS = f"{OUT_DIR}/1.1_{TAG}_Results_{DATE}.csv"
FILE_HP_IMPACT = f"{OUT_DIR}/1.2_{TAG}_HP_Impact_{DATE}.png"
FILE_HP_INTERACT = f"{OUT_DIR}/1.3_{TAG}_HP_Interaction_{DATE}.png"

# Figure styling
FS_SUPTITLE = 16
FS_TITLE = 14
FS_LABEL = 12
FS_TICK = 10
FIGSIZE_HP = (20, 10)
FIGSIZE_HEAT = (14, 5)

# ══════════════════════════════════════════════════════════════════════════════
# HYPERPARAMETER GRID
# ══════════════════════════════════════════════════════════════════════════════

param_grid = {
    'hidden_layer1': [512, 768, 1024],
    'hidden_layer2': [256, 512],
    'hidden_layer3': [64, 128, 256],
    'dropout_rate': [0.3, 0.4, 0.5],
    'l2_reg': [0.005, 0.01],
    'learning_rate': [1e-4, 1e-3],
    'batch_size': [32, 64],
}

# ══════════════════════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════

def print_header(title):
    """Print formatted header"""
    print("\n" + "="*80)
    print(title)
    print("="*80)

def print_step(step_num, title):
    """Print formatted step"""
    print(f"\n[STEP {step_num}] {title}")
    print("-" * 80)

def validate_data(X, y):
    """Validate data"""
    print_step(2, "VALIDATING DATA")
    
    if len(X) == 0 or len(y) == 0:
        print("❌ Empty dataset")
        return False
    
    x_nulls = X.isnull().sum().sum()
    y_nulls = y.isnull().sum()
    if x_nulls > 0 or y_nulls > 0:
        print(f"❌ NULL values found (X={x_nulls}, y={y_nulls})")
        return False
    
    x_inf = np.isinf(X.values).sum()
    x_nan = np.isnan(X.values).sum()
    if x_inf > 0 or x_nan > 0:
        print(f"❌ inf/NaN found (inf={x_inf}, nan={x_nan})")
        return False
    
    class_counts = y.value_counts().sort_index()
    print(f"✓ Class distribution: {dict(class_counts)}")
    
    if len(class_counts) != 2:
        print(f"❌ Expected 2 classes, found {len(class_counts)}")
        return False
    
    unique_labels = sorted(y.unique())
    if unique_labels != [0, 1]:
        print(f"❌ Labels must be [0, 1], found {unique_labels}")
        return False
    
    dup_count = X.duplicated().sum()
    if dup_count > 0:
        print(f"⚠️  Warning: {dup_count} duplicate rows found")
    
    print(f"✓ Data validation PASSED")
    return True

def create_mlp_model(hidden_layer1, hidden_layer2, hidden_layer3, 
                     dropout_rate, l2_reg, learning_rate, input_dim):
    """Create CryptoGuard-MLP with BatchNormalization"""
    model = Sequential([
        keras.layers.Input(shape=(input_dim,)),
        Dense(hidden_layer1, activation='relu',
              kernel_regularizer=regularizers.l2(l2_reg)),
        BatchNormalization(),
        Dropout(dropout_rate),
        
        Dense(hidden_layer2, activation='relu',
              kernel_regularizer=regularizers.l2(l2_reg)),
        BatchNormalization(),
        Dropout(dropout_rate),
        
        Dense(hidden_layer3, activation='relu',
              kernel_regularizer=regularizers.l2(l2_reg)),
        BatchNormalization(),
        Dropout(dropout_rate),
        
        Dense(1, activation='sigmoid',
              kernel_regularizer=regularizers.l2(l2_reg))
    ])
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

def get_callbacks(patience_es, patience_lr):
    """Get training callbacks"""
    callbacks = []
    if patience_es is not None:
        callbacks.append(
            EarlyStopping(monitor='val_loss', patience=patience_es,
                         restore_best_weights=True, verbose=0)
        )
    if patience_lr is not None:
        callbacks.append(
            ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                             patience=patience_lr, min_lr=1e-6, verbose=0)
        )
    return callbacks

def clear_memory():
    """Clear memory"""
    tf.keras.backend.clear_session()
    gc.collect()

def plot_hyperparameter_impact(results_df):
    """Plot hyperparameter sensitivity"""
    print("\nGenerating Hyperparameter Impact Plot...")
    
    fig, axes = plt.subplots(2, 3, figsize=FIGSIZE_HP)
    fig.suptitle('Hyperparameter Sensitivity Analysis', fontsize=FS_SUPTITLE)
    
    params_to_plot = [
        (axes[0, 0], 'param_hidden_layer1', 'Hidden Layer 1 Size', 'Number of Units'),
        (axes[0, 1], 'param_hidden_layer2', 'Hidden Layer 2 Size', 'Number of Units'),
        (axes[0, 2], 'param_hidden_layer3', 'Hidden Layer 3 Size', 'Number of Units'),
        (axes[1, 0], 'param_dropout_rate',  'Dropout Rate',        'Dropout Rate'),
        (axes[1, 1], 'param_l2_reg',        'L2 Regularization',   'L2 Weight'),
        (axes[1, 2], 'param_learning_rate', 'Learning Rate',       'Learning Rate'),
    ]
    
    for ax, param_col, title, xlabel in params_to_plot:
        if results_df[param_col].nunique() > 1:
            sns.boxplot(data=results_df, x=param_col, y='mean_test_score', ax=ax, showfliers=False)
        ax.set_title(title, fontsize=FS_TITLE)
        ax.set_xlabel(xlabel, fontsize=FS_LABEL)
        ax.set_ylabel('Mean CV MCC', fontsize=FS_LABEL)
        ax.tick_params(labelsize=FS_TICK)
    
    plt.tight_layout()
    plt.savefig(FILE_HP_IMPACT, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  ✓ Saved: {FILE_HP_IMPACT}")

def plot_hyperparameter_interaction(results_df):
    """Plot hyperparameter interactions"""
    print("Generating Hyperparameter Interaction Plot...")
    
    fig, axes = plt.subplots(1, 2, figsize=FIGSIZE_HEAT)
    fig.suptitle('Hyperparameter Interaction Analysis', fontsize=FS_SUPTITLE)
    
    # Heatmap 1: Dropout vs L2
    if results_df['param_dropout_rate'].nunique() > 1 and results_df['param_l2_reg'].nunique() > 1:
        pivot_dropout_l2 = results_df.pivot_table(
            values='mean_test_score',
            index='param_dropout_rate',
            columns='param_l2_reg',
            aggfunc='mean'
        )
        sns.heatmap(pivot_dropout_l2, annot=True, fmt='.4f', cmap='RdYlGn', ax=axes[0],
                   annot_kws={'size': FS_TICK},
                   vmin=results_df['mean_test_score'].min() - 0.002,
                   vmax=results_df['mean_test_score'].max() + 0.002)
    axes[0].set_title('Mean CV MCC: Dropout vs L2 Regularization', fontsize=FS_TITLE)
    axes[0].set_xlabel('L2 Regularization', fontsize=FS_LABEL)
    axes[0].set_ylabel('Dropout Rate', fontsize=FS_LABEL)
    
    # Heatmap 2: LR vs L2
    if results_df['param_learning_rate'].nunique() > 1 and results_df['param_l2_reg'].nunique() > 1:
        pivot_lr_l2 = results_df.pivot_table(
            values='mean_test_score',
            index='param_learning_rate',
            columns='param_l2_reg',
            aggfunc='mean'
        )
        sns.heatmap(pivot_lr_l2, annot=True, fmt='.4f', cmap='RdYlGn', ax=axes[1],
                   annot_kws={'size': FS_TICK},
                   vmin=results_df['mean_test_score'].min() - 0.002,
                   vmax=results_df['mean_test_score'].max() + 0.002)
    axes[1].set_title('Mean CV MCC: Learning Rate vs L2 Regularization', fontsize=FS_TITLE)
    axes[1].set_xlabel('L2 Regularization', fontsize=FS_LABEL)
    axes[1].set_ylabel('Learning Rate', fontsize=FS_LABEL)
    
    plt.tight_layout()
    plt.savefig(FILE_HP_INTERACT, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  ✓ Saved: {FILE_HP_INTERACT}")

# ══════════════════════════════════════════════════════════════════════════════
# MAIN GRIDSEARCH ENGINE
# ══════════════════════════════════════════════════════════════════════════════

def manual_grid_search(X, y, param_grid, input_dim):
    """Grid search with 5-fold CV + checkpoint system"""
    
    print_step(3, "GRIDSEARCH WITH 5-FOLD CV (432 COMBINATIONS)")
    
    param_names = list(param_grid.keys())
    param_values = list(param_grid.values())
    combos = list(product(*param_values))
    total = len(combos)
    
    print(f"Total combinations: {total}")
    print(f"CV folds: {CV_FOLDS}")
    print(f"Total runs: {total * CV_FOLDS:,}")
    
    # Checkpoint logic
    if os.path.exists(FILE_RESULTS):
        existing_df = pd.read_csv(FILE_RESULTS)
        start_idx = len(existing_df)
        results = existing_df.drop(columns=['rank_test_score'], errors='ignore').to_dict('records')
        print(f"⏸️  RESUMING FROM CHECKPOINT")
        print(f"✓ Found {start_idx} completed combos")
        print(f"➤ Restarting from combo {start_idx + 1}/{total}\n")
    else:
        results = []
        start_idx = 0
        print(f"▶️  STARTING FRESH (0/{total} completed)\n")
    
    skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
    t_start = time.time()
    
    for idx, combo in enumerate(combos, 1):
        if idx <= start_idx:
            continue
        
        params = dict(zip(param_names, combo))
        fold_mccs_train = []
        fold_mccs_test = []
        fold_times = []
        
        completed_pct = (idx / total) * 100
        print(f"[{idx:>3}/{total} ({completed_pct:5.1f}%)] "
              f"h1={params['hidden_layer1']} h2={params['hidden_layer2']} "
              f"h3={params['hidden_layer3']} drop={params['dropout_rate']} "
              f"l2={params['l2_reg']} lr={params['learning_rate']} bs={params['batch_size']}")
        
        for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
            t0 = time.time()
            
            scaler = StandardScaler()
            X_tr = scaler.fit_transform(X.iloc[tr_idx]).astype(np.float32)
            X_va = scaler.transform(X.iloc[va_idx]).astype(np.float32)
            y_tr = y.iloc[tr_idx].values.astype(np.float32)
            y_va = y.iloc[va_idx].values.astype(np.float32)
            
            model = create_mlp_model(
                hidden_layer1=params['hidden_layer1'],
                hidden_layer2=params['hidden_layer2'],
                hidden_layer3=params['hidden_layer3'],
                dropout_rate=params['dropout_rate'],
                l2_reg=params['l2_reg'],
                learning_rate=params['learning_rate'],
                input_dim=input_dim,
            )
            
            history = model.fit(
                X_tr, y_tr,
                epochs=EPOCHS,
                batch_size=params['batch_size'],
                validation_data=(X_va, y_va),
                callbacks=get_callbacks(PATIENCE_ES, PATIENCE_LR),
                verbose=0,
            )
            
            y_pred_tr = (model.predict(X_tr, verbose=0, batch_size=params['batch_size']) > 0.5).astype(int).flatten()
            y_pred_va = (model.predict(X_va, verbose=0, batch_size=params['batch_size']) > 0.5).astype(int).flatten()
            
            mcc_train = matthews_corrcoef(y_tr, y_pred_tr)
            mcc_val = matthews_corrcoef(y_va, y_pred_va)
            
            fold_mccs_train.append(mcc_train)
            fold_mccs_test.append(mcc_val)
            fold_times.append(time.time() - t0)
            
            del model, X_tr, X_va, y_tr, y_va, history
            clear_memory()
        
        mean_train = float(np.mean(fold_mccs_train))
        std_train = float(np.std(fold_mccs_train))
        mean_test = float(np.mean(fold_mccs_test))
        std_test = float(np.std(fold_mccs_test))
        gap_mcc = mean_train - mean_test
        mean_time = float(np.mean(fold_times))
        
        print(f"  >>> Train: {mean_train:.4f}±{std_train:.4f} | Val: {mean_test:.4f}±{std_test:.4f} | Gap: {gap_mcc:.4f}\n")
        
        results.append({
            'mean_fit_time': round(mean_time, 4),
            'std_fit_time': round(float(np.std(fold_times)), 4),
            'param_hidden_layer1': params['hidden_layer1'],
            'param_hidden_layer2': params['hidden_layer2'],
            'param_hidden_layer3': params['hidden_layer3'],
            'param_dropout_rate': params['dropout_rate'],
            'param_l2_reg': params['l2_reg'],
            'param_learning_rate': params['learning_rate'],
            'param_batch_size': params['batch_size'],
            'split0_train_score': fold_mccs_train[0],
            'split1_train_score': fold_mccs_train[1],
            'split2_train_score': fold_mccs_train[2],
            'split3_train_score': fold_mccs_train[3],
            'split4_train_score': fold_mccs_train[4],
            'mean_train_score': round(mean_train, 4),
            'std_train_score': round(std_train, 4),
            'split0_test_score': fold_mccs_test[0],
            'split1_test_score': fold_mccs_test[1],
            'split2_test_score': fold_mccs_test[2],
            'split3_test_score': fold_mccs_test[3],
            'split4_test_score': fold_mccs_test[4],
            'mean_test_score': round(mean_test, 4),
            'std_test_score': round(std_test, 4),
            'Gap_MCC': round(gap_mcc, 4),
        })
        
        # Save checkpoint
        df_tmp = pd.DataFrame(results)
        df_tmp['rank_test_score'] = df_tmp['mean_test_score'].rank(ascending=False, method='min').astype(int)
        col_order = ['mean_fit_time', 'std_fit_time', 'param_hidden_layer1', 'param_hidden_layer2',
                     'param_hidden_layer3', 'param_dropout_rate', 'param_l2_reg', 'param_learning_rate',
                     'param_batch_size', 'split0_train_score', 'split1_train_score', 'split2_train_score',
                     'split3_train_score', 'split4_train_score', 'mean_train_score', 'std_train_score',
                     'split0_test_score', 'split1_test_score', 'split2_test_score', 'split3_test_score',
                     'split4_test_score', 'mean_test_score', 'std_test_score', 'rank_test_score', 'Gap_MCC']
        df_tmp = df_tmp[col_order].sort_values('rank_test_score')
        df_tmp.to_csv(FILE_RESULTS, index=False)
        
        if idx % 50 == 0:
            elapsed = time.time() - t_start
            rate = (idx - start_idx) / elapsed if elapsed > 0 else 0
            remaining = (total - idx) / rate if rate > 0 else 0
            print(f"  ✓ Checkpoint: {idx}/{total} combos ({elapsed/3600:.1f}h elapsed, ~{remaining/3600:.1f}h remaining)\n")
    
    # Final ranking
    df_final = pd.DataFrame(results)
    df_final['rank_test_score'] = df_final['mean_test_score'].rank(ascending=False, method='min').astype(int)
    col_order = ['mean_fit_time', 'std_fit_time', 'param_hidden_layer1', 'param_hidden_layer2',
                 'param_hidden_layer3', 'param_dropout_rate', 'param_l2_reg', 'param_learning_rate',
                 'param_batch_size', 'split0_train_score', 'split1_train_score', 'split2_train_score',
                 'split3_train_score', 'split4_train_score', 'mean_train_score', 'std_train_score',
                 'split0_test_score', 'split1_test_score', 'split2_test_score', 'split3_test_score',
                 'split4_test_score', 'mean_test_score', 'std_test_score', 'rank_test_score', 'Gap_MCC']
    df_final = df_final[col_order].sort_values('rank_test_score').reset_index(drop=True)
    return df_final

# ══════════════════════════════════════════════════════════════════════════════
# MAIN EXECUTION
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    
    print_header("PART 1: D_CRYPTOGUARD-MLP GRIDSEARCH ENGINE")
    
    t_total_start = time.time()
    
    # PRE-FLIGHT CHECKS
    print_step("PRE", "PRE-FLIGHT CHECKS")
    print(f"✓ Python version: {sys.version.split()[0]}")
    print(f"✓ NumPy version: {np.__version__}")
    print(f"✓ Pandas version: {pd.__version__}")
    print(f"✓ TensorFlow version: {tf.__version__}")
    print(f"✓ Output directory: {OUT_DIR}")
    
    # LOAD DATA
    print_step(1, "LOADING DATA")
    df = pd.read_excel('./Datasets/D.xlsx')
    X = df.drop(columns=['CLASS_LABEL'])
    y = df['CLASS_LABEL']
    input_dim = X.shape[1]
    print(f"✓ X shape: {X.shape} (features)")
    print(f"✓ y shape: {y.shape} (labels)")
    print(f"✓ Input dimension: {input_dim}")
    
    # VALIDATE DATA
    if not validate_data(X, y):
        print("❌ Data validation failed")
        sys.exit(1)
    
    # GRIDSEARCH
    t_grid_start = time.time()
    results_df = manual_grid_search(X, y, param_grid, input_dim)
    t_grid = time.time() - t_grid_start
    
    print(f"\n✓ GridSearch completed: {len(results_df)} results")
    print(f"✓ Time: {t_grid/3600:.2f} hours")
    
    # DISPLAY RESULTS
    print_step(4, "GRIDSEARCH RESULTS")
    
    best = results_df.iloc[0]
    print("\n[STRATEGY 1] BEST VALIDATION MCC (Rank 1):")
    print(f"  Architecture: {int(best['param_hidden_layer1'])}-{int(best['param_hidden_layer2'])}-{int(best['param_hidden_layer3'])}")
    print(f"  CV Val MCC: {best['mean_test_score']:.4f} ± {best['std_test_score']:.4f}")
    
    min_gap_idx = results_df['Gap_MCC'].idxmin()
    min_gap = results_df.iloc[min_gap_idx]
    
    print("\n[STRATEGY 2] MIN GAP MCC (Best Generalization):")
    print(f"  Architecture: {int(min_gap['param_hidden_layer1'])}-{int(min_gap['param_hidden_layer2'])}-{int(min_gap['param_hidden_layer3'])}")
    print(f"  CV Val MCC: {min_gap['mean_test_score']:.4f} ± {min_gap['std_test_score']:.4f}")
    
    print("\nTop 10 Configurations (by Val MCC):")
    for i, (_, row) in enumerate(results_df.head(10).iterrows(), 1):
        print(f"{i:2}. {int(row['param_hidden_layer1'])}-{int(row['param_hidden_layer2'])}-{int(row['param_hidden_layer3'])} | "
              f"Val={row['mean_test_score']:.4f}")
    
    # GENERATE HYPERPARAMETER PLOTS
    print_step(5, "GENERATING HYPERPARAMETER PLOTS")
    
    plot_hyperparameter_impact(results_df)
    plot_hyperparameter_interaction(results_df)
    
    # FINAL SUMMARY
    t_total = time.time() - t_total_start
    
    print_header("✅ PART 1 COMPLETE!")
    
    print(f"\nExecution Summary:")
    print(f"  GridSearch Time: {t_grid/3600:.2f} hours")
    print(f"  Total Time: {t_total/3600:.2f} hours")
    print(f"  Date: {DATE}")
    
    print(f"\nOutput Files:")
    files = [
        (FILE_RESULTS, "GridSearch results (432 combos)"),
        (FILE_HP_IMPACT, "Hyperparameter sensitivity plot"),
        (FILE_HP_INTERACT, "Hyperparameter interaction plot"),
    ]
    
    for filepath, desc in files:
        exists = "✓" if os.path.exists(filepath) else "✗"
        print(f"  {exists} {os.path.basename(filepath)}")
        print(f"      ({desc})")
    
    print(f"\nOutput Directory: {OUT_DIR}")
    
    print_header("END OF PART 1")


PART 1: D_CRYPTOGUARD-MLP GRIDSEARCH ENGINE

[STEP PRE] PRE-FLIGHT CHECKS
--------------------------------------------------------------------------------
✓ Python version: 3.10.14
✓ NumPy version: 1.26.4
✓ Pandas version: 2.3.3
✓ TensorFlow version: 2.16.2
✓ Output directory: ./0. D_CryptoGuard-MLP_GridSearchCV_5Fold_08082026

[STEP 1] LOADING DATA
--------------------------------------------------------------------------------
✓ X shape: (20000, 525) (features)
✓ y shape: (20000,) (labels)
✓ Input dimension: 525

[STEP 2] VALIDATING DATA
--------------------------------------------------------------------------------
✓ Class distribution: {0: 10000, 1: 10000}
⚠️  Warning: 2 duplicate rows found
✓ Data validation PASSED

[STEP 3] GRIDSEARCH WITH 5-FOLD CV (432 COMBINATIONS)
--------------------------------------------------------------------------------
Total combinations: 432
CV folds: 5
Total runs: 2,160
⏸️  RESUMING FROM CHECKPOINT
✓ Found 432 completed combos
➤ Restarting from co

In [5]:
"""
================================================================================
PART 2A: D_CRYPTOGUARD-MLP BEST MCC EVALUATION
5-Fold CV with structured metrics table + Train/Val curves
================================================================================
"""

import os
import sys
import gc
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from keras import Sequential, regularizers
from keras.layers import Dense, Dropout, Input, BatchNormalization
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (matthews_corrcoef, accuracy_score, precision_score, 
                             recall_score, f1_score, confusion_matrix)
import matplotlib.pyplot as plt

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.config.set_visible_devices([], 'GPU')

DATASET = "D"
DATE = "08082026"
CV_FOLDS = 5
EPOCHS_FINAL = 150
PATIENCE_ES = 5
PATIENCE_LR = 5

FILE_RESULTS = "./0. D_CryptoGuard-MLP_GridSearchCV_5Fold_08082026/1.1_D_CryptoGuard-MLP_GridSearchCV_5Fold_Results_08082026.csv"

TAG = "D_CryptoGuard-MLP_GridSearchCV_5Fold"
OUT_DIR = f"./0. {TAG}_{DATE}"
os.makedirs(OUT_DIR, exist_ok=True)
FILE_LOSS_PLOT = f"{OUT_DIR}/2A.1_{TAG}_BestMCC_Training_Loss_{DATE}.png"
FILE_ACC_PLOT = f"{OUT_DIR}/2A.2_{TAG}_BestMCC_Training_Accuracy_{DATE}.png"
FILE_METRICS = f"{OUT_DIR}/2A.3_{TAG}_BestMCC_Evaluation_{DATE}.csv"

def print_header(title):
    print("\n" + "="*80)
    print(title)
    print("="*80)

def print_step(step_num, title):
    print(f"\n[STEP {step_num}] {title}")
    print("-" * 80)

def create_mlp_model(hidden_layer1, hidden_layer2, hidden_layer3, 
                     dropout_rate, l2_reg, learning_rate, input_dim):
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(hidden_layer1, activation='relu',
              kernel_regularizer=regularizers.l2(l2_reg)),
        BatchNormalization(),
        Dropout(dropout_rate),
        
        Dense(hidden_layer2, activation='relu',
              kernel_regularizer=regularizers.l2(l2_reg)),
        BatchNormalization(),
        Dropout(dropout_rate),
        
        Dense(hidden_layer3, activation='relu',
              kernel_regularizer=regularizers.l2(l2_reg)),
        BatchNormalization(),
        Dropout(dropout_rate),
        
        Dense(1, activation='sigmoid',
              kernel_regularizer=regularizers.l2(l2_reg))
    ])
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

def get_callbacks():
    return [
        EarlyStopping(monitor='val_loss', patience=PATIENCE_ES,
                     restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                         patience=PATIENCE_LR, min_lr=1e-6, verbose=0)
    ]

def calculate_metrics(y_true, y_pred):
    """Calculate all metrics"""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0
    
    return {
        'TP': int(tp),
        'FP': int(fp),
        'FN': int(fn),
        'TN': int(tn),
        'Accuracy': acc,
        'FPR': fpr,
        'FNR': fnr,
        'MCC': mcc,
        'Precision': prec,
        'Recall': rec,
        'F-measure': f1
    }

if __name__ == "__main__":
    
    print_header("PART 2A: D_CRYPTOGUARD-MLP BEST MCC 5-FOLD EVALUATION")
    
    # STEP 1: Load Part 1 results
    print_step(1, "LOADING PART 1 RESULTS")
    
    if not os.path.exists(FILE_RESULTS):
        print(f"❌ Error: Part 1 results not found")
        sys.exit(1)
    
    results_df = pd.read_csv(FILE_RESULTS)
    best = results_df.iloc[0]  # RANK 1: Highest Val MCC
    
    # Extract configuration
    h1 = int(best['param_hidden_layer1'])
    h2 = int(best['param_hidden_layer2'])
    h3 = int(best['param_hidden_layer3'])
    dr = float(best['param_dropout_rate'])
    l2r = float(best['param_l2_reg'])
    lr = float(best['param_learning_rate'])
    bs = int(best['param_batch_size'])
    cv_train_mcc = float(best['mean_train_score'])
    cv_val_mcc = float(best['mean_test_score'])
    cv_gap_mcc = float(best['Gap_MCC'])
    rank = int(best['rank_test_score'])
    
    print(f"\n✓ STRATEGY 1: Best MCC (Rank {rank})")
    print(f"  Architecture  : {h1}-{h2}-{h3}")
    print(f"  Dropout       : {dr}")
    print(f"  L2            : {l2r}")
    print(f"  LR            : {lr}")
    print(f"  Batch Size    : {bs}")
    print(f"  CV Train MCC  : {cv_train_mcc:.4f}")
    print(f"  CV Val MCC    : {cv_val_mcc:.4f}")
    print(f"  CV Gap MCC    : {cv_gap_mcc:.4f}")
    
    # STEP 2: Load data
    print_step(2, "LOADING FULL DATASET")
    
    df = pd.read_excel('./Datasets/D.xlsx')
    X = df.drop(columns=['CLASS_LABEL'])
    y = df['CLASS_LABEL']
    input_dim = X.shape[1]
    
    print(f"✓ X shape: {X.shape}")
    print(f"✓ y shape: {y.shape}")
    
    # STEP 3: 5-Fold CV with Stratified Train/Val/Test Split
    print_step(3, "5-FOLD STRATIFIED CV WITH TRAIN/VAL/TEST SPLIT")
    
    skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
    fold_metrics_list = []
    fold_histories = []
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
        print(f"\n[Fold {fold}/{CV_FOLDS}]")
        
        # Split 1: 5-fold stratified CV (80% train, 20% test)
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]
        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]
        
        # Split 2: Within training, stratified split (10% validation)
        X_train_real, X_val, y_train_real, y_val = train_test_split(
            X_train, y_train,
            test_size=0.10,
            random_state=SEED,
            stratify=y_train
        )
        
        # Scale (fit on actual training, transform val and test)
        scaler = StandardScaler()
        X_train_real_sc = scaler.fit_transform(X_train_real).astype(np.float32)
        X_val_sc = scaler.transform(X_val).astype(np.float32)
        X_test_sc = scaler.transform(X_test).astype(np.float32)
        y_train_real = y_train_real.values.astype(np.float32)
        y_val = y_val.values.astype(np.float32)
        y_test = y_test.values.astype(np.float32)
        
        # Create & train model
        model = create_mlp_model(h1, h2, h3, dr, l2r, lr, input_dim)
        
        history = model.fit(
            X_train_real_sc, y_train_real,
            epochs=EPOCHS_FINAL,
            batch_size=bs,
            validation_data=(X_val_sc, y_val),
            callbacks=get_callbacks(),
            verbose=0
        )
        
        # Collect training history for plotting
        fold_histories.append(history.history)
        
        # Track epochs trained (Early Stopping may stop before EPOCHS_FINAL)
        epochs_trained = len(history.history['loss'])
        
        # Predictions on test set
        y_pred_proba = model.predict(X_test_sc, verbose=0, batch_size=bs)
        y_pred = (y_pred_proba > 0.5).astype(int).flatten()
        
        # Calculate metrics
        metrics = calculate_metrics(y_test, y_pred)
        
        fold_metrics_list.append({
            'Fold': f'Fold {fold}',
            'Epochs_Trained': epochs_trained,
            'TP': metrics['TP'],
            'FP': metrics['FP'],
            'FN': metrics['FN'],
            'TN': metrics['TN'],
            'Accuracy': metrics['Accuracy'],
            'FPR': metrics['FPR'],
            'FNR': metrics['FNR'],
            'MCC': metrics['MCC'],
            'Precision': metrics['Precision'],
            'Recall': metrics['Recall'],
            'F-measure': metrics['F-measure'],
            'Best_H1': h1,
            'Best_H2': h2,
            'Best_H3': h3,
            'Best_Dropout': dr,
            'Best_L2': l2r,
            'Best_LR': lr,
            'Best_BatchSize': bs,
            'CV_Train_MCC': cv_train_mcc,
            'CV_Val_MCC': cv_val_mcc,
            'CV_Gap_MCC': cv_gap_mcc,
            'Rank': rank
        })
        
        print(f"  TP={metrics['TP']}, TN={metrics['TN']}, FP={metrics['FP']}, FN={metrics['FN']}")
        print(f"  Acc={metrics['Accuracy']:.4f}, FPR={metrics['FPR']:.4f}, FNR={metrics['FNR']:.4f}, MCC={metrics['MCC']:.4f}")
        print(f"  Prec={metrics['Precision']:.4f}, Rec={metrics['Recall']:.4f}, F1={metrics['F-measure']:.4f}")
        print(f"  Epochs Trained: {epochs_trained}/{EPOCHS_FINAL}")
        
        del model, X_train, X_test, X_train_real, X_val, X_train_real_sc, X_val_sc, X_test_sc, history
        tf.keras.backend.clear_session()
        gc.collect()
    
    # STEP 4: Calculate mean & std
    print_step(4, "CALCULATING MEAN ± STD")
    
    fold_metrics_df = pd.DataFrame(fold_metrics_list)
    
    # Create mean row
    mean_row = {
        'Fold': 'Mean',
        'TP': '-',
        'FP': '-',
        'FN': '-',
        'TN': '-',
        'Accuracy': f"{fold_metrics_df['Accuracy'].mean():.4f}±{fold_metrics_df['Accuracy'].std():.4f}",
        'FPR': f"{fold_metrics_df['FPR'].mean():.4f}±{fold_metrics_df['FPR'].std():.4f}",
        'FNR': f"{fold_metrics_df['FNR'].mean():.4f}±{fold_metrics_df['FNR'].std():.4f}",
        'MCC': f"{fold_metrics_df['MCC'].mean():.4f}±{fold_metrics_df['MCC'].std():.4f}",
        'Precision': f"{fold_metrics_df['Precision'].mean():.4f}±{fold_metrics_df['Precision'].std():.4f}",
        'Recall': f"{fold_metrics_df['Recall'].mean():.4f}±{fold_metrics_df['Recall'].std():.4f}",
        'F-measure': f"{fold_metrics_df['F-measure'].mean():.4f}±{fold_metrics_df['F-measure'].std():.4f}",
        'Best_H1': '',
        'Best_H2': '',
        'Best_H3': '',
        'Best_Dropout': '',
        'Best_L2': '',
        'Best_LR': '',
        'Best_BatchSize': '',
        'CV_Train_MCC': '',
        'CV_Val_MCC': '',
        'CV_Gap_MCC': '',
        'Rank': ''
    }
    
    # Combine fold data + mean row
    final_df = pd.concat([fold_metrics_df, pd.DataFrame([mean_row])], ignore_index=True)
    
    # STEP 5: Save CSV
    print_step(5, "SAVING EVALUATION CSV")
    
    final_df.to_csv(FILE_METRICS, index=False)
    print(f"✓ Saved: {FILE_METRICS}")
    
    print("\n✓ CSV Structure:")
    print("  Row 1: Headers")
    print("  Rows 2-6: Fold 1-5 metrics")
    print("  Row 7: Mean ± Std for Accuracy to F-measure")
    
    print("\n✓ Summary:")
    print(f"  Overall Accuracy: {fold_metrics_df['Accuracy'].mean():.4f} ± {fold_metrics_df['Accuracy'].std():.4f}")
    print(f"  Overall MCC: {fold_metrics_df['MCC'].mean():.4f} ± {fold_metrics_df['MCC'].std():.4f}")
    
    # Font sizes (user preference)
    FS_TITLE = 14
    FS_LABEL = 12
    FS_LEGEND = 12
    FS_TICK = 10
    
    # STEP 6: Generate training curves (mean ± std across 5 folds)
    print_step(6, "GENERATING TRAINING CURVES (TRAIN + VALIDATION)")
    
    # Use minimum epochs where ALL folds have valid data (no forward-fill)
    min_epochs = min([len(h['loss']) for h in fold_histories])
    print(f"\n✓ min_epochs across {len(fold_histories)} folds: {min_epochs}")
    
    # Collect only real data (no padding or forward-fill)
    train_loss_list = []
    val_loss_list = []
    train_acc_list = []
    val_acc_list = []
    
    for h in fold_histories:
        train_loss_list.append(h['loss'][:min_epochs])
        val_loss_list.append(h['val_loss'][:min_epochs])
        train_acc_list.append(h['accuracy'][:min_epochs])
        val_acc_list.append(h['val_accuracy'][:min_epochs])
    
    train_loss_array = np.array(train_loss_list)
    val_loss_array = np.array(val_loss_list)
    train_acc_array = np.array(train_acc_list)
    val_acc_array = np.array(val_acc_list)
    
    # Calculate mean and std
    train_loss_mean = np.mean(train_loss_array, axis=0)
    train_loss_std = np.std(train_loss_array, axis=0)
    val_loss_mean = np.mean(val_loss_array, axis=0)
    val_loss_std = np.std(val_loss_array, axis=0)
    
    train_acc_mean = np.mean(train_acc_array, axis=0)
    train_acc_std = np.std(train_acc_array, axis=0)
    val_acc_mean = np.mean(val_acc_array, axis=0)
    val_acc_std = np.std(val_acc_array, axis=0)
    
    epochs = np.arange(1, len(train_loss_mean) + 1)
    
    # Plot 1: Training Loss
    print("\nGenerating: Best Model Loss Plot...")
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(epochs, train_loss_mean, color='#FF0000', linewidth=1.5, label='Training Loss')
    ax.plot(epochs, val_loss_mean, color='#FF0000', linewidth=1.5, linestyle='--', label='Validation Loss')
    ax.fill_between(epochs,
                    train_loss_mean - train_loss_std,
                    train_loss_mean + train_loss_std,
                    color='#FF0000', alpha=0.3)
    ax.fill_between(epochs,
                    val_loss_mean - val_loss_std,
                    val_loss_mean + val_loss_std,
                    color='#FF0000', alpha=0.3)
    ax.set_title('CryptoGuard-MLP — Training and Validation Loss (μ ± σ Across 5 Folds)', fontsize=FS_TITLE, pad=12)
    ax.set_xlabel('Epoch', fontsize=FS_LABEL)
    ax.set_ylabel('Loss', fontsize=FS_LABEL)
    ax.tick_params(axis='both', labelsize=FS_TICK)
    ax.legend(fontsize=FS_LEGEND, loc='upper right', frameon=True, shadow=True)
    plt.tight_layout()
    plt.savefig(FILE_LOSS_PLOT, dpi=300, bbox_inches='tight')
    plt.clf()
    plt.close()
    print(f"  Saved → {FILE_LOSS_PLOT}")
    
    # Plot 2: Training Accuracy
    print("Generating: Best Model Accuracy Plot...")
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.plot(epochs, train_acc_mean, color='#228B22', linewidth=1.5, label='Training Accuracy')
    ax.plot(epochs, val_acc_mean, color='#228B22', linewidth=1.5, linestyle='--', label='Validation Accuracy')
    ax.fill_between(epochs,
                    train_acc_mean - train_acc_std,
                    train_acc_mean + train_acc_std,
                    color='#228B22', alpha=0.3)
    ax.fill_between(epochs,
                    val_acc_mean - val_acc_std,
                    val_acc_mean + val_acc_std,
                    color='#228B22', alpha=0.3)
    ax.set_title('CryptoGuard-MLP — Training and Validation Accuracy (μ ± σ Across 5 Folds)', fontsize=FS_TITLE, pad=12)
    ax.set_xlabel('Epoch', fontsize=FS_LABEL)
    ax.set_ylabel('Accuracy', fontsize=FS_LABEL)
    ax.tick_params(axis='both', labelsize=FS_TICK)
    ax.legend(fontsize=FS_LEGEND, loc='lower right', frameon=True, shadow=True)
    plt.tight_layout()
    plt.savefig(FILE_ACC_PLOT, dpi=300, bbox_inches='tight')
    plt.clf()
    plt.close()
    print(f"  Saved → {FILE_ACC_PLOT}")
    
    print_header("✅ PART 2A COMPLETE!")
    print(f"\nOutput Files:")
    print(f"  ✓ {FILE_LOSS_PLOT}")
    print(f"  ✓ {FILE_ACC_PLOT}")
    print(f"  ✓ {FILE_METRICS}")
    print(f"\nOutput Directory: {OUT_DIR}")
    print_header("END OF PART 2A")


PART 2A: D_CRYPTOGUARD-MLP BEST MCC 5-FOLD EVALUATION

[STEP 1] LOADING PART 1 RESULTS
--------------------------------------------------------------------------------

✓ STRATEGY 1: Best MCC (Rank 1)
  Architecture  : 768-512-128
  Dropout       : 0.5
  L2            : 0.005
  LR            : 0.0001
  Batch Size    : 64
  CV Train MCC  : 0.9075
  CV Val MCC    : 0.8689
  CV Gap MCC    : 0.0386

[STEP 2] LOADING FULL DATASET
--------------------------------------------------------------------------------
✓ X shape: (20000, 525)
✓ y shape: (20000,)

[STEP 3] 5-FOLD STRATIFIED CV WITH TRAIN/VAL/TEST SPLIT
--------------------------------------------------------------------------------

[Fold 1/5]
  TP=1874, TN=1847, FP=153, FN=126
  Acc=0.9303, FPR=0.0765, FNR=0.0630, MCC=0.8606
  Prec=0.9245, Rec=0.9370, F1=0.9307
  Epochs Trained: 65/150

[Fold 2/5]
  TP=1865, TN=1860, FP=140, FN=135
  Acc=0.9313, FPR=0.0700, FNR=0.0675, MCC=0.8625
  Prec=0.9302, Rec=0.9325, F1=0.9313
  Epochs Trained

In [7]:
"""
================================================================================
PART 2B: D_CRYPTOGUARD-MLP BEST MCC EVALUATION
All 5 Folds in 2 Combined Plots (Loss & Accuracy)
Universal Colors + Synced Fonts with Part 2A
================================================================================
"""

import os
import sys
import gc
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from keras import Sequential, regularizers
from keras.layers import Dense, Dropout, Input, BatchNormalization
from keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (matthews_corrcoef, accuracy_score, precision_score, 
                             recall_score, f1_score, confusion_matrix)
import matplotlib.pyplot as plt

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)
tf.config.set_visible_devices([], 'GPU')

DATASET = "D"
DATE = "08082026"
CV_FOLDS = 5
EPOCHS_FINAL = 100
PATIENCE_ES = 5
PATIENCE_LR = 5

FILE_RESULTS = "./0. D_CryptoGuard-MLP_GridSearchCV_5Fold_08082026/1.1_D_CryptoGuard-MLP_GridSearchCV_5Fold_Results_08082026.csv"

TAG = "D_CryptoGuard-MLP_GridSearchCV_5Fold"
OUT_DIR = f"./0. {TAG}_{DATE}"
os.makedirs(OUT_DIR, exist_ok=True)
FILE_LOSS_PLOT = f"{OUT_DIR}/2B.1_{TAG}_BestMCC_Training_Loss_{DATE}.png"
FILE_ACC_PLOT = f"{OUT_DIR}/2B.2_{TAG}_BestMCC_Training_Accuracy_{DATE}.png"
FILE_METRICS = f"{OUT_DIR}/2B.3_{TAG}_BestMCC_Evaluation_{DATE}.csv"


# ══════════════════════════════════════════════════════════════════════════════
# COLOR & FONT CONFIGURATION (SYNCED WITH PART 2A)
# ══════════════════════════════════════════════════════════════════════════════

# UNIVERSAL FOLD COLORS (Same for Loss & Accuracy)
FOLD_COLORS = {
    1: '#FF0000',  # Red
    2: '#FF8000',  # Orange
    3: '#FF00FF',  # Magenta/Pink
    4: '#0000FF',  # Blue
    5: '#7F00FF'   # Violet/Purple
}

FOLD_LABELS = {
    1: 'Fold #1',
    2: 'Fold #2',
    3: 'Fold #3',
    4: 'Fold #4',
    5: 'Fold #5'
}

# ══════════════════════════════════════════════════════════════════════════════
# COLOR & LINE STYLE MAPPING
# ══════════════════════════════════════════════════════════════════════════════
#
# Fold #1:
#   ├─ Train: Red solid line (─)
#   └─ Val: Red dashed line (- -)
#
# Fold #2:
#   ├─ Train: Orange solid line (─)
#   └─ Val: Orange dashed line (- -)
#
# Fold #3:
#   ├─ Train: Magenta solid line (─)
#   └─ Val: Magenta dashed line (- -)
#
# Fold #4:
#   ├─ Train: Blue solid line (─)
#   └─ Val: Blue dashed line (- -)
#
# Fold #5:
#   ├─ Train: Violet solid line (─)
#   └─ Val: Violet dashed line (- -)
#
# ══════════════════════════════════════════════════════════════════════════════

# FONT SIZES (SYNCED WITH PART 2A)
FS_TITLE = 14
FS_LABEL = 12
FS_LEGEND = 12
FS_TICK = 10

# LINE STYLES
LINE_WIDTH = 1.5
LINE_STYLE_TRAIN = '-'      # Solid (Train)
LINE_STYLE_VAL = '--'       # Dashed (Val)
ALPHA = 1.0

def print_header(title):
    print("\n" + "="*80)
    print(title)
    print("="*80)

def print_step(step_num, title):
    print(f"\n[STEP {step_num}] {title}")
    print("-" * 80)

def create_mlp_model(hidden_layer1, hidden_layer2, hidden_layer3, 
                     dropout_rate, l2_reg, learning_rate, input_dim):
    """Create CryptoGuard-MLP"""
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(hidden_layer1, activation='relu',
              kernel_regularizer=regularizers.l2(l2_reg)),
        BatchNormalization(),
        Dropout(dropout_rate),
        
        Dense(hidden_layer2, activation='relu',
              kernel_regularizer=regularizers.l2(l2_reg)),
        BatchNormalization(),
        Dropout(dropout_rate),
        
        Dense(hidden_layer3, activation='relu',
              kernel_regularizer=regularizers.l2(l2_reg)),
        BatchNormalization(),
        Dropout(dropout_rate),
        
        Dense(1, activation='sigmoid',
              kernel_regularizer=regularizers.l2(l2_reg))
    ])
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

def get_callbacks():
    """Get training callbacks"""
    return [
        EarlyStopping(monitor='val_loss', patience=PATIENCE_ES,
                     restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                         patience=PATIENCE_LR, min_lr=1e-6, verbose=0)
    ]

def calculate_metrics(y_true, y_pred):
    """Calculate all metrics"""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)
    
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0.0
    
    return {
        'TP': int(tp),
        'FP': int(fp),
        'FN': int(fn),
        'TN': int(tn),
        'Accuracy': acc,
        'FPR': fpr,
        'FNR': fnr,
        'MCC': mcc,
        'Precision': prec,
        'Recall': rec,
        'F-measure': f1
    }

if __name__ == "__main__":
    
    print_header("PART 2B: D_CRYPTOGUARD-MLP ALL FOLDS IN 2 COMBINED PLOTS")
    
    # STEP 1: Load Part 1 results
    print_step(1, "LOADING PART 1 RESULTS")
    
    if not os.path.exists(FILE_RESULTS):
        print(f"❌ Error: Part 1 results not found")
        sys.exit(1)
    
    results_df = pd.read_csv(FILE_RESULTS)
    best = results_df.iloc[0]
    
    h1 = int(best['param_hidden_layer1'])
    h2 = int(best['param_hidden_layer2'])
    h3 = int(best['param_hidden_layer3'])
    dr = float(best['param_dropout_rate'])
    l2r = float(best['param_l2_reg'])
    lr = float(best['param_learning_rate'])
    bs = int(best['param_batch_size'])
    cv_train_mcc = float(best['mean_train_score'])
    cv_val_mcc = float(best['mean_test_score'])
    cv_gap_mcc = float(best['Gap_MCC'])
    rank = int(best['rank_test_score'])
    
    print(f"\n✓ STRATEGY 1: Best MCC (Rank {rank})")
    print(f"  Architecture  : {h1}-{h2}-{h3}")
    print(f"  Dropout       : {dr}")
    print(f"  L2            : {l2r}")
    print(f"  LR            : {lr}")
    print(f"  Batch Size    : {bs}")
    print(f"  CV Train MCC  : {cv_train_mcc:.4f}")
    print(f"  CV Val MCC    : {cv_val_mcc:.4f}")
    print(f"  CV Gap MCC    : {cv_gap_mcc:.4f}")
    
    # STEP 2: Load data
    print_step(2, "LOADING FULL DATASET")
    
    df = pd.read_excel('./Datasets/D.xlsx')
    X = df.drop(columns=['CLASS_LABEL'])
    y = df['CLASS_LABEL']
    input_dim = X.shape[1]
    
    print(f"✓ X shape: {X.shape}")
    print(f"✓ y shape: {y.shape}")
    
    # STEP 3: 5-Fold CV - Collect all fold histories
    print_step(3, "5-FOLD STRATIFIED CV - COLLECTING FOLD HISTORIES")
    
    skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
    fold_metrics_list = []
    fold_histories = {}
    
    for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
        print(f"\n[Fold {fold}/{CV_FOLDS}]")
        
        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]
        y_train = y.iloc[train_idx]
        y_test = y.iloc[test_idx]
        
        X_train_real, X_val, y_train_real, y_val = train_test_split(
            X_train, y_train,
            test_size=0.10,
            random_state=SEED,
            stratify=y_train
        )
        
        scaler = StandardScaler()
        X_train_real_sc = scaler.fit_transform(X_train_real).astype(np.float32)
        X_val_sc = scaler.transform(X_val).astype(np.float32)
        X_test_sc = scaler.transform(X_test).astype(np.float32)
        y_train_real = y_train_real.values.astype(np.float32)
        y_val = y_val.values.astype(np.float32)
        y_test = y_test.values.astype(np.float32)
        
        model = create_mlp_model(h1, h2, h3, dr, l2r, lr, input_dim)
        
        history = model.fit(
            X_train_real_sc, y_train_real,
            epochs=EPOCHS_FINAL,
            batch_size=bs,
            validation_data=(X_val_sc, y_val),
            callbacks=get_callbacks(),
            verbose=0
        )
        
        epochs_trained = len(history.history['loss'])
        
        fold_histories[fold] = {
            'epochs': np.arange(1, epochs_trained + 1),
            'train_loss': history.history['loss'],
            'val_loss': history.history['val_loss'],
            'train_acc': history.history['accuracy'],
            'val_acc': history.history['val_accuracy']
        }
        
        y_pred_proba = model.predict(X_test_sc, verbose=0, batch_size=bs)
        y_pred = (y_pred_proba > 0.5).astype(int).flatten()
        
        metrics = calculate_metrics(y_test, y_pred)
        
        fold_metrics_list.append({
            'Fold': f'Fold {fold}',
            'Epochs_Trained': epochs_trained,
            'TP': metrics['TP'],
            'FP': metrics['FP'],
            'FN': metrics['FN'],
            'TN': metrics['TN'],
            'Accuracy': metrics['Accuracy'],
            'FPR': metrics['FPR'],
            'FNR': metrics['FNR'],
            'MCC': metrics['MCC'],
            'Precision': metrics['Precision'],
            'Recall': metrics['Recall'],
            'F-measure': metrics['F-measure'],
            'Best_H1': h1,
            'Best_H2': h2,
            'Best_H3': h3,
            'Best_Dropout': dr,
            'Best_L2': l2r,
            'Best_LR': lr,
            'Best_BatchSize': bs,
            'CV_Train_MCC': cv_train_mcc,
            'CV_Val_MCC': cv_val_mcc,
            'CV_Gap_MCC': cv_gap_mcc,
            'Rank': rank
        })
        
        print(f"  TP={metrics['TP']}, TN={metrics['TN']}, FP={metrics['FP']}, FN={metrics['FN']}")
        print(f"  Acc={metrics['Accuracy']:.4f}, FPR={metrics['FPR']:.4f}, FNR={metrics['FNR']:.4f}, MCC={metrics['MCC']:.4f}")
        print(f"  Prec={metrics['Precision']:.4f}, Rec={metrics['Recall']:.4f}, F1={metrics['F-measure']:.4f}")
        print(f"  Epochs Trained: {epochs_trained}/{EPOCHS_FINAL}")
        
        del model, X_train, X_test, X_train_real, X_val, X_train_real_sc, X_val_sc, X_test_sc, history
        tf.keras.backend.clear_session()
        gc.collect()
    
    # STEP 4: Calculate mean & std
    print_step(4, "CALCULATING MEAN ± STD")
    
    fold_metrics_df = pd.DataFrame(fold_metrics_list)
    
    mean_row = {
        'Fold': 'Mean',
        'Epochs_Trained': '-',
        'TP': '-',
        'FP': '-',
        'FN': '-',
        'TN': '-',
        'Accuracy': f"{fold_metrics_df['Accuracy'].mean():.4f}±{fold_metrics_df['Accuracy'].std():.4f}",
        'FPR': f"{fold_metrics_df['FPR'].mean():.4f}±{fold_metrics_df['FPR'].std():.4f}",
        'FNR': f"{fold_metrics_df['FNR'].mean():.4f}±{fold_metrics_df['FNR'].std():.4f}",
        'MCC': f"{fold_metrics_df['MCC'].mean():.4f}±{fold_metrics_df['MCC'].std():.4f}",
        'Precision': f"{fold_metrics_df['Precision'].mean():.4f}±{fold_metrics_df['Precision'].std():.4f}",
        'Recall': f"{fold_metrics_df['Recall'].mean():.4f}±{fold_metrics_df['Recall'].std():.4f}",
        'F-measure': f"{fold_metrics_df['F-measure'].mean():.4f}±{fold_metrics_df['F-measure'].std():.4f}",
        'Best_H1': '',
        'Best_H2': '',
        'Best_H3': '',
        'Best_Dropout': '',
        'Best_L2': '',
        'Best_LR': '',
        'Best_BatchSize': '',
        'CV_Train_MCC': '',
        'CV_Val_MCC': '',
        'CV_Gap_MCC': '',
        'Rank': ''
    }
    
    final_df = pd.concat([fold_metrics_df, pd.DataFrame([mean_row])], ignore_index=True)
    
    # STEP 5: Save CSV
    print_step(5, "SAVING EVALUATION CSV")
    
    final_df.to_csv(FILE_METRICS, index=False)
    print(f"✓ Saved: {FILE_METRICS}")
    
    print("\n✓ CSV Structure:")
    print("  Row 1: Headers")
    print("  Rows 2-6: Fold 1-5 metrics")
    print("  Row 7: Mean ± Std for Accuracy to F-measure")
    
    print("\n✓ Summary:")
    print(f"  Overall Accuracy: {fold_metrics_df['Accuracy'].mean():.4f} ± {fold_metrics_df['Accuracy'].std():.4f}")
    print(f"  Overall MCC: {fold_metrics_df['MCC'].mean():.4f} ± {fold_metrics_df['MCC'].std():.4f}")
    
    # STEP 6: Generate Combined Plots
    print_step(6, "GENERATING COMBINED FOLD PLOTS (LOSS & ACCURACY)")
    
    # ========================================================================
    # PLOT 1: ALL FOLDS - LOSS (Train Solid + Val Dashed)
    # ========================================================================
    
    print("\nGenerating: All Folds Loss Plot...")
    fig, ax = plt.subplots(figsize=(14, 7))
    
    for fold in range(1, CV_FOLDS + 1):
        epochs = fold_histories[fold]['epochs']
        train_loss = fold_histories[fold]['train_loss']
        val_loss = fold_histories[fold]['val_loss']
        color = FOLD_COLORS[fold]
        label = FOLD_LABELS[fold]
        
        # Train: Solid line (show label only for legend)
        ax.plot(epochs, train_loss, color=color, linewidth=LINE_WIDTH, 
                linestyle=LINE_STYLE_TRAIN, alpha=ALPHA, label=label)
        
        # Val: Dashed line (NO label)
        ax.plot(epochs, val_loss, color=color, linewidth=LINE_WIDTH, 
                linestyle=LINE_STYLE_VAL, alpha=ALPHA)
    
    # Add dummy lines for Train and Val styles
    ax.plot([], [], color='black', linewidth=LINE_WIDTH, 
            linestyle=LINE_STYLE_TRAIN, label='Train')
    ax.plot([], [], color='black', linewidth=LINE_WIDTH, 
            linestyle=LINE_STYLE_VAL, label='Val')
    
    ax.set_title('CryptoGuard-MLP - Training and Validation Loss (5-Fold CV) on D', 
                fontsize=FS_TITLE, pad=15)
    ax.set_xlabel('Epoch', fontsize=FS_LABEL)
    ax.set_ylabel('Loss', fontsize=FS_LABEL)
    ax.tick_params(axis='both', labelsize=FS_TICK)
    ax.legend(fontsize=FS_LEGEND, loc='upper right', frameon=True, shadow=True, ncol=1)
    ax.grid(True, alpha=0.3, linestyle=':')
    
    plt.tight_layout()
    plt.savefig(FILE_LOSS_PLOT, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  ✓ Saved: {os.path.basename(FILE_LOSS_PLOT)}")
    
    # ========================================================================
    # PLOT 2: ALL FOLDS - ACCURACY (Train Solid + Val Dashed)
    # ========================================================================
    
    print("Generating: All Folds Accuracy Plot...")
    fig, ax = plt.subplots(figsize=(14, 7))
    
    for fold in range(1, CV_FOLDS + 1):
        epochs = fold_histories[fold]['epochs']
        train_acc = fold_histories[fold]['train_acc']
        val_acc = fold_histories[fold]['val_acc']
        color = FOLD_COLORS[fold]
        label = FOLD_LABELS[fold]
        
        # Train: Solid line (show label only for legend)
        ax.plot(epochs, train_acc, color=color, linewidth=LINE_WIDTH, 
                linestyle=LINE_STYLE_TRAIN, alpha=ALPHA, label=label)
        
        # Val: Dashed line (NO label)
        ax.plot(epochs, val_acc, color=color, linewidth=LINE_WIDTH, 
                linestyle=LINE_STYLE_VAL, alpha=ALPHA)
    
    # Add dummy lines for Train and Val styles
    ax.plot([], [], color='black', linewidth=LINE_WIDTH, 
            linestyle=LINE_STYLE_TRAIN, label='Train')
    ax.plot([], [], color='black', linewidth=LINE_WIDTH, 
            linestyle=LINE_STYLE_VAL, label='Val')
    
    ax.set_title('CryptoGuard-MLP - Training and Validation Accuracy (5-Fold CV) on D', 
                fontsize=FS_TITLE, pad=15)
    ax.set_xlabel('Epoch', fontsize=FS_LABEL)
    ax.set_ylabel('Accuracy', fontsize=FS_LABEL)
    ax.tick_params(axis='both', labelsize=FS_TICK)
    ax.legend(fontsize=FS_LEGEND, loc='lower right', frameon=True, shadow=True, ncol=1)
    ax.grid(True, alpha=0.3, linestyle=':')
    
    plt.tight_layout()
    plt.savefig(FILE_ACC_PLOT, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"  ✓ Saved: {os.path.basename(FILE_ACC_PLOT)}")
    
    print_header("✅ PART 2B COMPLETE!")
    print(f"\nOutput Directory: {OUT_DIR}")
    print(f"\nGenerated Files:")
    print(f"  ✓ 2B.1_{TAG}_AllFolds_Loss_{DATE}.png")
    print(f"  ✓ 2B.2_{TAG}_AllFolds_Accuracy_{DATE}.png")
    print(f"  ✓ 2B.0_{TAG}_Evaluation_{DATE}.csv")
    print(f"\nColor & Line Style Configuration:")
    print(f"  Fold #1: #FF0000 (Red) - Solid (Train) & Dashed (Val)")
    print(f"  Fold #2: #FF8000 (Orange) - Solid (Train) & Dashed (Val)")
    print(f"  Fold #3: #FF00FF (Magenta) - Solid (Train) & Dashed (Val)")
    print(f"  Fold #4: #0000FF (Blue) - Solid (Train) & Dashed (Val)")
    print(f"  Fold #5: #7F00FF (Violet) - Solid (Train) & Dashed (Val)")
    print(f"\nLegend Configuration:")
    print(f"  7 items: Fold #1-5 + Train + Val")
    print(f"  Line Width: {LINE_WIDTH}")
    print(f"  Alpha: {ALPHA}")
    print(f"\nFont Sizes (Synced with Part 2A):")
    print(f"  Title: {FS_TITLE}")
    print(f"  Label: {FS_LABEL}")
    print(f"  Legend: {FS_LEGEND}")
    print(f"  Tick: {FS_TICK}")
    print_header("END OF PART 2B")


PART 2B: D_CRYPTOGUARD-MLP ALL FOLDS IN 2 COMBINED PLOTS

[STEP 1] LOADING PART 1 RESULTS
--------------------------------------------------------------------------------

✓ STRATEGY 1: Best MCC (Rank 1)
  Architecture  : 768-512-128
  Dropout       : 0.5
  L2            : 0.005
  LR            : 0.0001
  Batch Size    : 64
  CV Train MCC  : 0.9075
  CV Val MCC    : 0.8689
  CV Gap MCC    : 0.0386

[STEP 2] LOADING FULL DATASET
--------------------------------------------------------------------------------
✓ X shape: (20000, 525)
✓ y shape: (20000,)

[STEP 3] 5-FOLD STRATIFIED CV - COLLECTING FOLD HISTORIES
--------------------------------------------------------------------------------

[Fold 1/5]
  TP=1874, TN=1847, FP=153, FN=126
  Acc=0.9303, FPR=0.0765, FNR=0.0630, MCC=0.8606
  Prec=0.9245, Rec=0.9370, F1=0.9307
  Epochs Trained: 65/100

[Fold 2/5]
  TP=1865, TN=1860, FP=140, FN=135
  Acc=0.9313, FPR=0.0700, FNR=0.0675, MCC=0.8625
  Prec=0.9302, Rec=0.9325, F1=0.9313
  Epochs Tr